# Chapter 5 &mdash; State Names as Residues: MSB-First "Divisible by 3"

**Concept 7 of the Chapter 5 decomposition:** *State Names as Residues: MSB-First "Divisible by 3"*

Track $N \bmod 3$, not $N$; the recurrence $N \mapsto 2N+b$ drives every transition.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Residue-States-MSB/Concept-Residue-States-MSB.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Read a binary numeral **most-significant bit first**. Appending bit $b$ maps the value
$N$ to $2N+b$.

You cannot store $N$ &mdash; it is unbounded. You can store $N \bmod 3$, because

$$(2N+b) \bmod 3 = (2(N \bmod 3) + b) \bmod 3.$$

Three states, one per residue, and the transition table is just that formula
evaluated six times. The state name **is** the residue.

## 2. Definitions

### The recurrence, as a function

In [ ]:
def step_resid(r, b, m=3): return (2*r + int(b)) % m
print("residue table (state, bit) -> state:")
for r in range(3):
    for b in '01':
        print("  (%d, %s) -> %d" % (r, b, step_resid(r, b)))

### The DFA, read straight off that table

In [ ]:
Div3 = md2mc('''DFA
IF : 0 -> IF     !! (2*0+0)%3 = 0
IF : 1 -> S1     !! (2*0+1)%3 = 1
S1 : 0 -> S2     !! (2*1+0)%3 = 2
S1 : 1 -> IF     !! (2*1+1)%3 = 0
S2 : 0 -> S1     !! (2*2+0)%3 = 1
S2 : 1 -> S2     !! (2*2+1)%3 = 2
''')

### The reference specification

In [ ]:
def in_Div3(s): return s != '' and int(s, 2) % 3 == 0

## 3. Tests

The state name is literally $N \bmod 3$.

In [ ]:
tag = {'IF': 0, 'S1': 1, 'S2': 2}
for s in ['', '0', '1', '11', '110', '1001', '10101']:
    v = int(s, 2) if s else 0
    print("%-8r value %-4d N%%3 = %d   state %s" % (s, v, v % 3, run_dfa(Div3, s)))
assert all(tag[run_dfa(Div3, s)] == (int(s,2) if s else 0) % 3
           for s in ['', '0', '1', '11', '110', '1001', '10101', '111111'])

So acceptance is divisibility &mdash; on every numeral up to 12 bits.

In [ ]:
from itertools import product
bad = [''.join(p) for k in range(1, 13) for p in product('01', repeat=k)
       if accepts_dfa(Div3, ''.join(p)) != (int(''.join(p), 2) % 3 == 0)]
print("mismatches on all 1..12-bit numerals :", bad)
assert not bad

Three states handle numerals of any size, including ones Python prints in scientific ranges.

In [ ]:
big = bin(3 * 7**40)[2:]
print("numeral of %d bits, divisible by 3? %s" % (len(big), accepts_dfa(Div3, big)))
assert accepts_dfa(Div3, big)
print("|Q| = %d, regardless." % len(Div3["Q"]))

## 4. Animation

Three residues, six edges &mdash; the whole of modular arithmetic in a picture.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(Div3, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Build the divisible-by-5 machine the same way. How many states?
2. Why does reading MSB-first make the recurrence so simple?
3. Which state would be final for "leaves remainder 1"?

In [ ]:
# Your work for the exercises above.